# Attention, from zero. Numpy only. Every number printed.

No torch. No libraries at all except numpy. No black boxes, no "trust me". Every calculation is
written out and printed, and every number in the text is printed by the cell above it.

**You need to know three things already:**

1. how to multiply and add,
2. what `e^x` is (and its derivative is `e^x` — so calculus is covered),
3. that a derivative means "if I nudge this number a little, how much does the answer move?"

That's it. If a word looks scary, it is defined the moment it appears.

**The plan — one tiny idea per cell**

| cell | idea |
|---|---|
| 2 | a vector is a list of numbers; **dot product** = "how much do these two agree?" |
| 3 | **softmax** = turn any numbers into percentages that add to 100% |
| 4 | words become vectors (the embedding table) |
| 5 | **attention for one word**, every step written out |
| 6 | the same for all words at once → the attention matrix, and what it changes |
| 7 | positions: why the model needs to know word order |
| 8 | **multi-head**: two different ways of asking, averaged |
| 9 | feed-forward, residual, LayerNorm — the rest of the block |
| 10 | stack the blocks, then turn the final vector into a word |
| 11 | **learning**: the derivative, done by hand, teaches the model *where to look* |
| 12 | the whole thing on one page, and what is honestly missing |

The running example the whole way through: **`the cat sat`**. We will watch the word `sat` slowly
fill up with the meaning of `cat`.

_State: done. Runs top to bottom, ~2 s, numpy is the only import. Key printed results: `sat` moves
from 0.71 to 0.90 alignment with `cat`; order-blindness shown as two identical matrices; LayerNorm's
2-dimension quirk left visible; learning takes attention on `cat` from 33% to 97% using nothing but a
finite-difference slope._


In [1]:
import numpy as np

# ---- tiny helpers so the output is readable (all plain python, no extra libraries) ----
def show(name, a, dec=3):
    a = np.asarray(a, dtype=float)
    print(f"{name:26s} {str(a.shape):10s} = {np.array2string(a, precision=dec, suppress_small=True)}")

def bars(values, names, width=38):
    m = max(abs(float(v)) for v in values) or 1.0
    for n, v in zip(names, values):
        print(f"     {n:>5s} {float(v):+.3f}  {'#' * int(round(abs(v) / m * width))}")

def curve(vals, height=10):
    vals = [float(v) for v in vals]
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        hi = lo + 1
    grid = [[" "] * len(vals) for _ in range(height)]
    for x, v in enumerate(vals):
        grid[height - 1 - int(round((v - lo) / (hi - lo) * (height - 1)))][x] = "*"
    for row in grid:
        print("      |" + "".join(row))
    print("      +" + "-" * len(vals))
    print(f"       step 0{' ' * max(0, len(vals) - 14)}step {len(vals) - 1}")
    print(f"       highest = {hi:.4f}   lowest = {lo:.4f}")

print("numpy is loaded. Nothing else is.")

# ---- 1. A VECTOR IS JUST A LIST OF NUMBERS ---------------------------------------------
# We describe a word by a few numbers. Two numbers here so you can check by eye;
# a real model uses 512, 4096, whatever.
the = np.array([1.0, 0.0])
cat = np.array([0.0, 1.0])
sat = np.array([1.0, 1.0])

show("'the' as numbers", the)
show("'cat' as numbers", cat)
show("'sat' as numbers", sat)

# ---- 2. THE DOT PRODUCT: "how much do these two agree?" --------------------------------
# Multiply matching numbers, add up. That is the whole operation.
def dot(a, b):
    total = 0.0
    for i in range(len(a)):
        print(f"        {a[i]:+.2f} * {b[i]:+.2f} = {a[i] * b[i]:+.3f}")
        total += a[i] * b[i]
    return total

print("\n  dot('the', 'cat')  -- step by step:")
d = dot(the, cat)
print(f"        sum = {d:+.3f}   <- zero: they have nothing in common\n")

print("  agreement between every pair:")
for na, a in [("the", the), ("cat", cat), ("sat", sat)]:
    for nb, b in [("the", the), ("cat", cat), ("sat", sat)]:
        print(f"     {na:>4s} . {nb:<4s} = {float(a @ b):+.3f}")
    print()

print("  Read it like this: a big positive number = 'we are pointing the same way',")
print("  zero = 'unrelated', negative = 'opposite'. THIS is the only way the model")
print("  will ever decide which words are relevant to which.")

numpy is loaded. Nothing else is.
'the' as numbers           (2,)       = [1. 0.]
'cat' as numbers           (2,)       = [0. 1.]
'sat' as numbers           (2,)       = [1. 1.]

  dot('the', 'cat')  -- step by step:
        +1.00 * +0.00 = +0.000
        +0.00 * +1.00 = +0.000
        sum = +0.000   <- zero: they have nothing in common

  agreement between every pair:
      the . the  = +1.000
      the . cat  = +0.000
      the . sat  = +1.000

      cat . the  = +0.000
      cat . cat  = +1.000
      cat . sat  = +1.000

      sat . the  = +1.000
      sat . cat  = +1.000
      sat . sat  = +2.000

  Read it like this: a big positive number = 'we are pointing the same way',
  zero = 'unrelated', negative = 'opposite'. THIS is the only way the model
  will ever decide which words are relevant to which.


In [2]:
# ---- 3. SOFTMAX: numbers into percentages ---------------------------------------------
# Problem: agreement scores can be any number at all (negative, 37.2, whatever).
# We need "how much should I listen to each word", i.e. shares that add to 100%.
# Recipe: raise e to each score, then divide by the sum.
def softmax(scores):
    scores = np.asarray(scores, dtype=float)
    e = np.exp(scores - scores.max())     # the -max is only to keep numbers small;
    return e / e.sum()                    # it does not change the answer at all

raw = np.array([1.0, 2.0, 3.0])
print("scores:", raw, "\n")
print("  e to each score:")
for s in raw:
    print(f"        e^{s:.1f} = {np.exp(s):.4f}")
total = np.exp(raw).sum()
print(f"        sum      = {total:.4f}\n")
print("  divide each by the sum:")
for s in raw:
    print(f"        {np.exp(s):.4f} / {total:.4f} = {np.exp(s) / total:.4f}")
print()
show("softmax(scores)", softmax(raw))
print(f"  do they add to 1?  {softmax(raw).sum():.12f}\n")
bars(softmax(raw), ["score 1", "score 2", "score 3"])

print("\n  Three properties, all you need to remember:")
print("    * everything comes out positive")
print("    * they always add up to exactly 1  (so: percentages)")
print("    * bigger score in -> bigger share out, and the difference is exaggerated")

print("\n  Two more things worth seeing:")
show("softmax([0,0,0])", softmax([0, 0, 0]))
print("     equal scores -> equal shares. This matters: an untrained model is")
print("     basically this, and it is why 'attention means nothing yet' at step 0.")
show("softmax([2,4,6])", softmax([2, 4, 6]))
print(f"     doubling the scores sharpens the choice: {softmax(raw)[0]:.3f}/{softmax(raw)[1]:.3f}/"
      f"{softmax(raw)[2]:.3f} becomes {softmax([2,4,6])[0]:.3f}/{softmax([2,4,6])[1]:.3f}/"
      f"{softmax([2,4,6])[2]:.3f}")
print("     That knob is called temperature, and it will come back later.")

print("\n  That is softmax. Attention is literally this, applied to agreement scores.")

scores: [1. 2. 3.] 

  e to each score:
        e^1.0 = 2.7183
        e^2.0 = 7.3891
        e^3.0 = 20.0855
        sum      = 30.1929

  divide each by the sum:
        2.7183 / 30.1929 = 0.0900
        7.3891 / 30.1929 = 0.2447
        20.0855 / 30.1929 = 0.6652

softmax(scores)            (3,)       = [0.09  0.245 0.665]
  do they add to 1?  1.000000000000

     score 1 +0.090  #####
     score 2 +0.245  ##############
     score 3 +0.665  ######################################

  Three properties, all you need to remember:
    * everything comes out positive
    * they always add up to exactly 1  (so: percentages)
    * bigger score in -> bigger share out, and the difference is exaggerated

  Two more things worth seeing:
softmax([0,0,0])           (3,)       = [0.333 0.333 0.333]
     equal scores -> equal shares. This matters: an untrained model is
     basically this, and it is why 'attention means nothing yet' at step 0.
softmax([2,4,6])           (3,)       = [0.016 0.117 0.

In [3]:
# ---- 4. WORDS BECOME NUMBERS -----------------------------------------------------------
# The model never sees letters. It sees rows of a lookup table, called embeddings.
# Think of each number as a dial describing a property. We do not know what the
# properties are, and nobody writes them down -- they are LEARNED (see cell 11).
# For this lesson I picked them by hand so the arithmetic stays small enough to check.
vocab = {
    "the": np.array([1.0, 0.0]),
    "cat": np.array([0.0, 1.0]),
    "dog": np.array([0.2, 1.0]),
    "sat": np.array([1.0, 1.0]),
}
words = ["the", "cat", "sat"]
ids = [0, 1, 3]        # the model's view: just indices

print("  word    index    the numbers")
for w, i in zip(["the", "cat", "dog", "sat"], [0, 1, 2, 3]):
    print(f"  {w:>4s}    {i:>3d}     {vocab[w]}")

print("\n  our sentence 'the cat sat' is this to the model:", ids)
X = np.array([vocab[w] for w in words])
show("X (one row per word)", X)

# Why these numbers? Because 'cat' and 'dog' should be more alike than 'cat' and 'the'.
def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

print("\n  how alike are words, using the dot product (scaled to -1..+1)?")
for a, b in [("cat", "dog"), ("cat", "the"), ("cat", "sat"), ("the", "sat")]:
    print(f"     cos({a:>3s}, {b:>3s}) = {cos(vocab[a], vocab[b]):+.3f}")

print("\n  So the dials already encode 'cat and dog are the same kind of thing'.")
print("  That is all an embedding is. Nothing magic has happened yet.")
print("\n  And this is 2 numbers per word. A real model uses 512 to 8192. Same idea.")
print("  Everything below is identical arithmetic, just with longer rows.")

  word    index    the numbers
   the      0     [1. 0.]
   cat      1     [0. 1.]
   dog      2     [0.2 1. ]
   sat      3     [1. 1.]

  our sentence 'the cat sat' is this to the model: [0, 1, 3]
X (one row per word)       (3, 2)     = [[1. 0.]
 [0. 1.]
 [1. 1.]]

  how alike are words, using the dot product (scaled to -1..+1)?
     cos(cat, dog) = +0.981
     cos(cat, the) = +0.000
     cos(cat, sat) = +0.707
     cos(the, sat) = +0.707

  So the dials already encode 'cat and dog are the same kind of thing'.
  That is all an embedding is. Nothing magic has happened yet.

  And this is 2 numbers per word. A real model uses 512 to 8192. Same idea.
  Everything below is identical arithmetic, just with longer rows.


In [4]:
# ---- 5. ATTENTION, FOR ONE WORD, EVERY STEP WRITTEN OUT --------------------------------
# The question we are answering for the word 'sat':
#   "of all the words in this sentence, which should I take meaning from?"
#
# Three characters in this story, and each is just a matrix we multiply by:
#   Query  Wq : what I am looking for      ("I'm a verb, find me a thing")
#   Key    Wk : what I am advertising      ("this is what I am")
#   Value  Wv : what I hand over if picked ("take my numbers")
#
# A real model LEARNS these matrices. Here I chose them by hand so you can see
# what they do; cell 11 shows learning happening for real.
Wq = np.array([[-1.0, 0.0],
               [ 0.0, 1.0]])     # "flip the first dial, keep the second"
Wk = np.array([[ 1.0, 0.0],
               [ 0.0, 1.0]])     # do nothing: advertise myself as I am
Wv = np.array([[ 1.0, 0.0],
               [ 0.0, 1.0]])     # do nothing: hand over myself

def matvec(name, M, x):
    """Multiply a matrix by a vector, printing every product."""
    print(f"   {name}:")
    out = np.zeros(M.shape[0])
    for r in range(M.shape[0]):
        terms = []
        for c in range(M.shape[1]):
            terms.append(f"({M[r, c]:+.0f} * {x[c]:+.1f})")
            out[r] += M[r, c] * x[c]
        print(f"      row {r+1}: {' + '.join(terms)} = {out[r]:+.3f}")
    return out

x_sat = vocab["sat"]
print("Our word is 'sat', which is the vector", x_sat, "\n")

q = matvec("1) Query: q = Wq * x_sat   <- the question 'sat' is asking", Wq, x_sat)
show("   q", q)

print("\n2) Keys: what every word advertises (here Wk does nothing, so key = its numbers)")
K = np.array([Wk @ vocab[w] for w in words])
for w, k in zip(words, K):
    print(f"      key({w:>3s}) = {k}")

print("\n3) Agreement between the question and each key (dot product, step by step for 'cat'):")
d_cat = dot(q, K[1])
scores = K @ q
print(f"\n      all three scores = {scores}")

print("\n4) Divide by sqrt(2) = 1.414, so the numbers do not get huge:")
scaled = scores / np.sqrt(2)
print(f"      {scores}  /  1.414  =  {scaled}")

print("\n5) Softmax: turn scores into shares that add to 1")
w = softmax(scaled)
for name, s, share in zip(words, scaled, w):
    print(f"      {name:>3s}: score {s:+.3f} -> share {share:.3f}")
print(f"      shares add up to {w.sum():.3f}")
bars(w, words)
print("\n      In words: 'sat' listens 58% to 'cat', 28% to itself, 14% to 'the'.")

print("\n6) Take that much of each word's numbers, and add them up:")
V = np.array([Wv @ vocab[n] for n in words])
total = np.zeros(2)
for name, share, v in zip(words, w, V):
    part = share * v
    total += part
    print(f"      {share:.3f} * {name:>3s} {v} = {part}")
print(f"      -----------------------------------------")
print(f"      new vector for 'sat'            = {total}")
show("\n   'sat' before", x_sat)
show("   'sat' after", total)

print(f"\n   how alike is 'sat' to 'cat' now?")
print(f"      before: {cos(x_sat, vocab['cat']):+.3f}")
print(f"      after : {cos(total, vocab['cat']):+.3f}   <- it moved towards 'cat'")

print("\n=== That is the whole of attention. Four lines: ===")
print("      q     = Wq @ x")
print("      score = (K @ q) / sqrt(d)          # agreement with every word")
print("      w     = softmax(score)             # shares that add to 1")
print("      out   = w @ V                      # take that share of each word")

Our word is 'sat', which is the vector [1. 1.] 

   1) Query: q = Wq * x_sat   <- the question 'sat' is asking:
      row 1: (-1 * +1.0) + (+0 * +1.0) = -1.000
      row 2: (+0 * +1.0) + (+1 * +1.0) = +1.000
   q                       (2,)       = [-1.  1.]

2) Keys: what every word advertises (here Wk does nothing, so key = its numbers)
      key(the) = [1. 0.]
      key(cat) = [0. 1.]
      key(sat) = [1. 1.]

3) Agreement between the question and each key (dot product, step by step for 'cat'):
        -1.00 * +0.00 = -0.000
        +1.00 * +1.00 = +1.000

      all three scores = [-1.  1.  0.]

4) Divide by sqrt(2) = 1.414, so the numbers do not get huge:
      [-1.  1.  0.]  /  1.414  =  [-0.70710678  0.70710678  0.        ]

5) Softmax: turn scores into shares that add to 1
      the: score -0.707 -> share 0.140
      cat: score +0.707 -> share 0.576
      sat: score +0.000 -> share 0.284
      shares add up to 1.000
       the +0.140  #########
       cat +0.576  ################

In [5]:
# ---- 6. ALL WORDS AT ONCE: the attention matrix -----------------------------------------
# Cell 5 did it for 'sat'. Now every word asks its own question, at the same time.
# Nothing new happens: it is the same four lines, run three times, stacked into a table.
def attention(X, Wq, Wk, Wv):
    Q = X @ Wq.T                     # one question per word      (3, 2)
    K = X @ Wk.T                     # one advertisement per word (3, 2)
    V = X @ Wv.T                     # one payload per word       (3, 2)
    scores = Q @ K.T / np.sqrt(X.shape[1])          # every question vs every ad -> (3, 3)
    W = np.array([softmax(row) for row in scores])  # each ROW becomes shares
    out = W @ V                      # each word collects its mix  (3, 2)
    return out, W, scores

out, W, scores = attention(X, Wq, Wk, Wv)

print("shapes:")
print(f"   X       {X.shape}   (3 words, 2 numbers each)")
print(f"   scores  {scores.shape}   (3 questions, 3 answers)  <- the big table")
print(f"   weights {W.shape}   same table, but every row turned into shares")
print(f"   out     {out.shape}   (3 words, 2 numbers each)  <- same shape as X\n")

print("THE ATTENTION MATRIX.  row = who is asking,  column = who they listen to\n")
print("            " + "".join(f"{w:>9s}" for w in words))
for i, name in enumerate(words):
    cells = "".join(f"{W[i, j]:9.3f}" for j in range(3))
    print(f"   {name:>6s} {cells}   | row sum = {W[i].sum():.3f}")

print("\n   The same thing as bars, one row at a time:")
for i, name in enumerate(words):
    print(f"\n   '{name}' listens to:")
    bars(W[i], words, width=30)

print("\n   Every row is different, because every word asks a different question:")
print(f"     'sat' -> {W[2,1]:.0%} of its attention on 'cat'   (a verb hunting for a subject)")
print(f"     'the' -> {W[0,1]:.0%} of its attention on 'cat'")
print(f"     'cat' -> splits {W[1,1]:.0%} itself / {W[1,2]:.0%} 'sat'")
print("   No rule anywhere says 'verbs should look at nouns'. It fell out of the")
print("   dot products, and in cell 11 you will see the model LEARN which way to look.")

print("\nTHE RESULT: every word is now a mix of the words it listened to.")
show("   X before attention", X)
show("   X after  attention", out)
print("\n   Same shape. That is what lets you stack this again and again:")
print("   a block takes 3 vectors in and gives 3 vectors out.")

print("\n   Row by row:")
for i, name in enumerate(words):
    print(f"      {name:>3s}: {X[i]} -> {out[i]}")
print("\n   Look at 'sat' (the last row): it was [1, 1] and is now pulled towards 'cat'.")

shapes:
   X       (3, 2)   (3 words, 2 numbers each)
   scores  (3, 3)   (3 questions, 3 answers)  <- the big table
   weights (3, 3)   same table, but every row turned into shares
   out     (3, 2)   (3 words, 2 numbers each)  <- same shape as X

THE ATTENTION MATRIX.  row = who is asking,  column = who they listen to

                  the      cat      sat
      the     0.248    0.503    0.248   | row sum = 1.000
      cat     0.198    0.401    0.401   | row sum = 1.000
      sat     0.140    0.576    0.284   | row sum = 1.000

   The same thing as bars, one row at a time:

   'the' listens to:
       the +0.248  ###############
       cat +0.503  ##############################
       sat +0.248  ###############

   'cat' listens to:
       the +0.198  ###############
       cat +0.401  ##############################
       sat +0.401  ##############################

   'sat' listens to:
       the +0.140  #######
       cat +0.576  ##############################
       sat +0.284 

In [6]:
# ---- 7. WORD ORDER: why positions have to be added -------------------------------------
# Try the sentence 'sat cat the' -- the same three words, different order.
order = [2, 1, 0]
X_other = X[order]
_, W_other, _ = attention(X_other, Wq, Wk, Wv)

print("attention for 'the cat sat':")
print(np.round(W, 3))
print("\nattention for 'sat cat the':")
print(np.round(W_other, 3))
print("\nwith the rows and columns put back in the same order:")
print(np.round(W[np.ix_(order, order)], 3))
print("\n   IDENTICAL. The model cannot tell the two sentences apart. Everything it")
print("   knows is 'these three vectors are here together', with no idea what came")
print("   first. That is a real bug if you care about meaning.")

# Fix: give each SLOT its own little vector and add it on.
pos = np.array([[0.0, 0.0],      # slot 1
                [0.1, 0.1],      # slot 2
                [0.2, 0.2]])     # slot 3
print("\n   position stickers (one per slot):")
for i, p in enumerate(pos):
    print(f"      slot {i+1}: {p}")

X_pos = X + pos
print("\n   add them to the word vectors:")
for i, w in enumerate(words):
    print(f"      {w:>3s}: {X[i]} + {pos[i]} = {X_pos[i]}")
show("\n   X with positions", X_pos)

_, W_pos, _ = attention(X_pos, Wq, Wk, Wv)
print("\nattention with positions added:")
print(np.round(W_pos, 3))
print("\ndifference from the version without positions:")
print(np.round(W_pos - W, 3))
print("\n   Now the numbers are different, so the model CAN tell the order apart.")
print("   Two identical words in different slots also get different stickers, which")
print("   is how a model knows which 'the' you mean.")

print("\n   Real models use 512 of these per slot, often smooth waves (sines and")
print("   cosines) or a learned table. Either way: one vector, added on, per slot.")
print("   From here on we always add them.")

attention for 'the cat sat':
[[0.248 0.503 0.248]
 [0.198 0.401 0.401]
 [0.14  0.576 0.284]]

attention for 'sat cat the':
[[0.284 0.576 0.14 ]
 [0.401 0.401 0.198]
 [0.248 0.503 0.248]]

with the rows and columns put back in the same order:
[[0.284 0.576 0.14 ]
 [0.401 0.401 0.198]
 [0.248 0.503 0.248]]

   IDENTICAL. The model cannot tell the two sentences apart. Everything it
   knows is 'these three vectors are here together', with no idea what came
   first. That is a real bug if you care about meaning.

   position stickers (one per slot):
      slot 1: [0. 0.]
      slot 2: [0.1 0.1]
      slot 3: [0.2 0.2]

   add them to the word vectors:
      the: [1. 0.] + [0. 0.] = [1. 0.]
      cat: [0. 1.] + [0.1 0.1] = [0.1 1.1]
      sat: [1. 1.] + [0.2 0.2] = [1.2 1.2]

   X with positions       (3, 2)     = [[1.  0. ]
 [0.1 1.1]
 [1.2 1.2]]

attention with positions added:
[[0.266 0.503 0.231]
 [0.166 0.417 0.417]
 [0.114 0.621 0.266]]

difference from the version without positions:


In [7]:
# ---- 8. MULTI-HEAD: several ways of asking, at the same time ----------------------------
# One set of (Wq, Wk, Wv) can only express one kind of relationship. So we run a few
# in parallel. Each is called a head. Two heads here, side by side:
Wq1, Wk1, Wv1 = Wq, Wk, Wv                       # head 1: "flip the first dial"
Wq2 = np.array([[1.0, 0.0],
                [0.0, 0.0]])                      # head 2: "only look at the first dial"
Wk2, Wv2 = Wk, Wv

head1, W1, _ = attention(X_pos, Wq1, Wk1, Wv1)
head2, W2h, _ = attention(X_pos, Wq2, Wk2, Wv2)

print("head 1 weights (what each word listens to):")
print(np.round(W1, 3))
print("\nhead 2 weights -- same words, different question, different answer:")
print(np.round(W2h, 3))
print("\nthe two heads disagree the most on 'sat':")
print(f"   head 1 gives 'cat' {W1[2,1]:.3f} and 'sat' {W1[2,2]:.3f}")
print(f"   head 2 gives 'cat' {W2h[2,1]:.3f} and 'sat' {W2h[2,2]:.3f}")

# Each head produced its own (3,2) result. Glue them side by side -> (3,4).
concat = np.concatenate([head1, head2], axis=1)
show("\n   head 1 output", head1)
show("   head 2 output", head2)
show("   glued together (3 words, 4 numbers)", concat)

# Then mix them back down to 2 numbers with a matrix Wo. Here: plain average.
Wo = 0.5 * np.array([[1.0, 0.0, 1.0, 0.0],
                     [0.0, 1.0, 0.0, 1.0]])
show("   Wo (average the two heads)", Wo)
multi = concat @ Wo.T
show("   result of multi-head attention", multi)

print("\n   For 'sat':", f"{head1[2]} (head 1) and {head2[2]} (head 2)")
print(f"             -> {multi[2]}  (average)")
print("\n   In a real model Wo is not an average -- it is another learned matrix, and")
print("   there are usually 8 to 64 heads. The shape of the computation is exactly")
print("   what you just did: several views, glued, mixed.")

head 1 weights (what each word listens to):
[[0.266 0.503 0.231]
 [0.166 0.417 0.417]
 [0.114 0.621 0.266]]

head 2 weights -- same words, different question, different answer:
[[0.373 0.197 0.43 ]
 [0.339 0.318 0.344]
 [0.377 0.176 0.447]]

the two heads disagree the most on 'sat':
   head 1 gives 'cat' 0.621 and 'sat' 0.266
   head 2 gives 'cat' 0.176 and 'sat' 0.447

   head 1 output          (3, 2)     = [[0.594 0.83 ]
 [0.708 0.959]
 [0.495 1.001]]
   head 2 output           (3, 2)     = [[0.908 0.733]
 [0.783 0.762]
 [0.931 0.73 ]]
   glued together (3 words, 4 numbers) (3, 4)     = [[0.594 0.83  0.908 0.733]
 [0.708 0.959 0.783 0.762]
 [0.495 1.001 0.931 0.73 ]]
   Wo (average the two heads) (2, 4)     = [[0.5 0.  0.5 0. ]
 [0.  0.5 0.  0.5]]
   result of multi-head attention (3, 2)     = [[0.751 0.782]
 [0.745 0.86 ]
 [0.713 0.866]]

   For 'sat': [0.49456464 1.00148133] (head 1) and [0.93120675 0.72974767] (head 2)
             -> [0.7128857 0.8656145]  (average)

   In a real

In [8]:
# ---- 9. THE REST OF THE BLOCK: residual, LayerNorm, feed-forward ------------------------
# Attention lets words TALK to each other. The rest of the block is about not losing
# what you had, and letting each word think on its own.

# 9a. RESIDUAL: keep the old vector, add the new information to it.
kept = X_pos + multi                    # "x + attention(x)"
print("9a. residual  x + attention(x)")
show("   x (with positions)", X_pos)
show("   attention output", multi)
show("   sum", kept)
print("   Why: attention only has to learn a CORRECTION, not rebuild the vector from")
print("   scratch. Without this, deep stacks do not train at all.")

# 9b. LAYERNORM: rescale each word's numbers to a sane range.
#     Formula: (x - mean) / spread.  Real models also multiply by a learned scale
#     and add a learned shift; we leave those at 1 and 0 here.
def layernorm(x):
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + 1e-8
    return (x - mu) / sd

print("\n9b. LayerNorm, arithmetic for the word 'sat':")
row = kept[2]
mu, sd = row.mean(), row.std()
print(f"      numbers        {row}")
print(f"      mean           {mu:+.4f}")
print(f"      spread (std)   {sd:+.4f}")
for i in range(len(row)):
    print(f"      ({row[i]:+.4f} - {mu:+.4f}) / {sd:+.4f} = {(row[i] - mu) / sd:+.4f}")
normed = layernorm(kept)
show("   after LayerNorm (all words)", normed)
print("      Notice what just broke: 'cat' and 'sat' came out IDENTICAL.")
print("      With only 2 numbers, LayerNorm keeps only the ORDER of the two (which")
print("      is bigger), so words with the same order collapse together. That is an")
print("      artifact of using 2 numbers to make the arithmetic checkable -- with 32")
print("      or 512 numbers this does not happen. It is worth knowing that tiny")
print("      demos have quirks like this: fewer dimensions = more coincidences.")
print("      So from here we carry the un-normalised 'kept' forward, to keep the")
print("      three words distinguishable. A real model keeps both steps.")

# 9c. FEED-FORWARD: each word, alone, through a little 2-layer network.
W1 = np.array([[1.0, 0.5],
               [-0.5, 1.0]])
b1 = np.array([0.0, 0.0])
W2 = np.array([[0.5, 0.0],
               [0.0, 0.5]])
print("\n9c. feed-forward on the word 'sat' (input = [1, 2]):")
x_in = np.array([1.0, 2.0])
h_pre = x_in @ W1.T + b1
print(f"      step 1: x * W1      = {h_pre}")
h = np.maximum(h_pre, 0.0)
print(f"      step 2: max(0, it)  = {h}      <- that is ReLU, the whole nonlinearity")
print(f"      step 3: h * W2      = {h @ W2.T}")
ff_all = np.maximum(kept @ W1.T + b1, 0.0) @ W2.T
show("\n   feed-forward output (all words, from 'kept')", ff_all)

block_out = kept + ff_all                   # second residual
show("\n9d. one block done: kept + feed-forward", block_out)
print("      Three words in, three words out, same shape, and still distinct.")

print("\n   Why a feed-forward at all? Attention mixes words, and mixing is linear: run")
print("   twenty mixes in a row and you still have one linear map. The ReLU is what")
print("   lets the model compute something new instead of only averaging. Roughly")
print("   two thirds of a real model's parameters live in these networks.")

9a. residual  x + attention(x)
   x (with positions)      (3, 2)     = [[1.  0. ]
 [0.1 1.1]
 [1.2 1.2]]
   attention output        (3, 2)     = [[0.751 0.782]
 [0.745 0.86 ]
 [0.713 0.866]]
   sum                     (3, 2)     = [[1.751 0.782]
 [0.845 1.96 ]
 [1.913 2.066]]
   Why: attention only has to learn a CORRECTION, not rebuild the vector from
   scratch. Without this, deep stacks do not train at all.

9b. LayerNorm, arithmetic for the word 'sat':
      numbers        [1.9128857 2.0656145]
      mean           +1.9893
      spread (std)   +0.0764
      (+1.9129 - +1.9893) / +0.0764 = -1.0000
      (+2.0656 - +1.9893) / +0.0764 = +1.0000
   after LayerNorm (all words) (3, 2)     = [[ 1. -1.]
 [-1.  1.]
 [-1.  1.]]
      Notice what just broke: 'cat' and 'sat' came out IDENTICAL.
      With only 2 numbers, LayerNorm keeps only the ORDER of the two (which
      is bigger), so words with the same order collapse together. That is an
      artifact of using 2 numbers to make the ari

In [9]:
# ---- 10. STACK THE BLOCKS, THEN TURN THE NUMBERS INTO A WORD ----------------------------
# A block = attention + residual + feed-forward + residual, all with the same shape in
# and out. Because the shape is preserved, you can stack them. Let's write it once and
# run it twice.
def block(x, Wq, Wk, Wv, W1, W2):
    att, _, _ = attention(x, Wq, Wk, Wv)     # words talk to each other
    x = x + att                              # residual
    ff = np.maximum(x @ W1.T, 0.0) @ W2.T    # each word thinks alone
    return x + ff                            # residual

x = X_pos
print("two blocks, same code, run twice:")
for n in range(2):
    x = block(x, Wq, Wk, Wv, W1, W2)
    show(f"   after block {n+1}", x)

# Now the output layer: turn the final vector of the LAST word into a score per word.
out_words = ["le", "chat", "assis"]          # what it could say next
W_out = np.array([[ 1.0,  0.0],              # hand-picked for the lesson
                  [ 0.0,  1.0],
                  [-1.0, -1.0]])
b_out = np.zeros(3)

print("\nthe output layer reads the LAST word's vector and scores every possible word:")
last = x[-1]
print(f"   last word ('sat') vector = {last}")
scores = W_out @ last + b_out
for wname, s in zip(out_words, scores):
    print(f"      score({wname:>5s}) = {s:+.3f}")
probs = softmax(scores)
print()
bars(probs, out_words)
print(f"\n   The model's 'answer' is the word with the biggest share: "
      f"'{out_words[int(probs.argmax())]}' at {probs.max():.1%}")

print("\n   Every position does this, not just the last (the same matrices are used):")
for i, name in enumerate(words):
    p = softmax(W_out @ x[i] + b_out)
    print(f"      after {name:>3s}: le {p[0]:.2f} | chat {p[1]:.2f} | assis {p[2]:.2f}")

print("\n=== That is a transformer. All of it. ===")
print("   words -> +positions -> [attention, residual, feed-forward, residual] x N -> scores")
print("   The only thing left is that every matrix above was picked by ME, by hand.")
print("   Cell 11 hands the picking over to arithmetic.")

two blocks, same code, run twice:
   after block 1           (3, 2)     = [[2.598 0.847]
 [1.727 2.886]
 [3.092 2.879]]
   after block 2           (3, 2)     = [[7.608 4.323]
 [6.947 7.725]
 [8.769 7.418]]

the output layer reads the LAST word's vector and scores every possible word:
   last word ('sat') vector = [8.76853197 7.41832084]
      score(   le) = +8.769
      score( chat) = +7.418
      score(assis) = -16.187

        le +0.794  ######################################
      chat +0.206  ##########
     assis +0.000  

   The model's 'answer' is the word with the biggest share: 'le' at 79.4%

   Every position does this, not just the last (the same matrices are used):
      after the: le 0.96 | chat 0.04 | assis 0.00
      after cat: le 0.31 | chat 0.69 | assis 0.00
      after sat: le 0.79 | chat 0.21 | assis 0.00

=== That is a transformer. All of it. ===
   words -> +positions -> [attention, residual, feed-forward, residual] x N -> scores
   The only thing left is that ever

In [10]:
# ---- 11. LEARNING: let arithmetic pick the numbers --------------------------------------
# Goal: we want the word 'sat' to look at 'cat'. Forget hand-picking a query --
# start from zero and let calculus find it.
#
# The only pieces we need:
#   1. a number that says how wrong we are   ->  loss = -ln(share on 'cat')
#   2. the derivative: nudge q slightly, see how the loss moves
#   3. step downhill, repeat
K_keys = X_pos                      # the three words (with positions) as keys
target = 1                          # index of 'cat'
def loss_and_shares(q):
    p = softmax((K_keys @ q) / np.sqrt(2))
    return -np.log(p[target] + 1e-12), p

def gradient(q, h=1e-4):
    """d(loss)/d(q_i)  by the definition of derivative -- no library, no chain rule."""
    g = np.zeros_like(q)
    for i in range(len(q)):
        up, down = q.copy(), q.copy()
        up[i] += h
        down[i] -= h
        g[i] = (loss_and_shares(up)[0] - loss_and_shares(down)[0]) / (2 * h)
    return g

q = np.array([0.0, 0.0])
L0, p0 = loss_and_shares(q)
print(f"start: q = {q}  ->  shares {np.round(p0, 3)}  ->  loss = {L0:.4f}")
print(f"       (loss = -ln(1/3) = ln(3) = {np.log(3):.4f}: with a zero query every word")
print("        gets an equal share, so this is the worst it can do)\n")

print("the derivative, worked out for the first step:")
for i in range(2):
    up, down = q.copy(), q.copy()
    up[i] += 1e-4
    down[i] -= 1e-4
    print(f"   nudge q[{i}] by +0.0001: loss {loss_and_shares(up)[0]:.6f}")
    print(f"   nudge q[{i}] by -0.0001: loss {loss_and_shares(down)[0]:.6f}")
    print(f"   slope = (up - down) / 0.0002 = {gradient(q)[i]:+.4f}\n")

lr = 1.5
losses, q = [], np.array([0.0, 0.0])
print("now walk downhill 40 times:  q  <-  q - 1.5 * slope\n")
print(f"{'step':>5s} {'loss':>8s} {'share on cat':>13s}   q")
for step in range(41):
    L, p = loss_and_shares(q)
    losses.append(L)
    if step % 5 == 0:
        print(f"{step:>5d} {L:>8.4f} {p[target]:>13.3f}   {np.round(q, 3)}")
    q = q - lr * gradient(q)

print("\nloss curve (each column is one step):")
curve(losses)

L, p = loss_and_shares(q)
print(f"\nfinal: q = {np.round(q, 3)}   loss = {L:.4f}   share on 'cat' = {p[target]:.1%}")
bars(p, words)
print(f"\nIt listens to 'cat' {p[target]:.0%}, up from 33%. Nobody told it to -- the slope did.")

print("\nSo what did it actually learn? Look at the score against each word:")
for wname, k in zip(words, K_keys):
    print(f"   q . key({wname:>3s}) = {float(q @ k):+.3f}")
print("   'cat' is the only word scoring above zero, so it wins. Note this is NOT")
print("   'pointing at cat' in any simple sense -- many different q would give this")
print("   same answer, and the walk just happened to reach this one.")

print(f"\n   Also note |q| grew from 0 to {np.linalg.norm(q):.2f} and the loss is still")
print("   creeping down at step 40. Nothing here told it to stop: a bigger q always")
print("   sharpens softmax, so the loss keeps falling. Real training adds weight decay")
print("   and normalisation for exactly this reason, and attention divides by sqrt(d)")
print("   to stop the scores running away. Every guard rail in a real model exists")
print("   because something like this happened to somebody.")

print("\n   And the matrix? A real model learns Wq, not q, with q = Wq @ x. Four numbers")
print("   instead of two, adjusted by exactly the rule you just used. Every weight in")
print("   a real model -- billions of them -- is nudged by that same slope, with the")
print("   chain rule telling it how much each one is to blame. Computing all those")
print("   slopes fast is the only thing a library like PyTorch adds.")

start: q = [0. 0.]  ->  shares [0.333 0.333 0.333]  ->  loss = 1.0986
       (loss = -ln(1/3) = ln(3) = 1.0986: with a zero query every word
        gets an equal share, so this is the worst it can do)

the derivative, worked out for the first step:
   nudge q[0] by +0.0001: loss 1.098659
   nudge q[0] by -0.0001: loss 1.098565
   slope = (up - down) / 0.0002 = +0.4714

   nudge q[1] by +0.0001: loss 1.098589
   nudge q[1] by -0.0001: loss 1.098636
   slope = (up - down) / 0.0002 = -0.2357

now walk downhill 40 times:  q  <-  q - 1.5 * slope

 step     loss  share on cat   q
    0   1.0986         0.333   [0. 0.]
    5   0.2475         0.781   [-2.325  0.996]
   10   0.1255         0.882   [-3.248  1.296]
   15   0.0829         0.920   [-3.797  1.46 ]
   20   0.0617         0.940   [-4.186  1.57 ]
   25   0.0490         0.952   [-4.486  1.652]
   30   0.0406         0.960   [-4.73   1.718]
   35   0.0347         0.966   [-4.936  1.772]
   40   0.0302         0.970   [-5.113  1.818]

lo

## The whole thing on one page

Six formulas. That is the complete vocabulary of a transformer — everything else is these, repeated.

```
1.  x          = embedding[word] + position                    a word becomes numbers
2.  q, k, v    = Wq·x,  Wk·x,  Wv·x                            three views of the same word
3.  score      = (q · k) / sqrt(d)                             how much do I care about you
4.  weight     = softmax(score)                                shares that add to 1
5.  attn_out   = sum of (weight × v)                            take that share of your meaning
6.  block(x)   = x + attn(x)   then   x + feedforward(x)        keep what you had, add what's new
    loss       = -ln(share given to the right answer)           how wrong am I
    learn      = weight  -=  lr × slope(loss, weight)            walk downhill
```

**The story of `sat` in this notebook:** `[1, 1]` → it looked at `cat` (58%) → became `[0.424, 0.860]`,
which is 0.90 aligned with `cat` where it used to be 0.71. Nothing moved between words except
numbers multiplied and added.

## What you now know, in plain words

- **A word is a list of numbers.** Similar words get similar lists.
- **Attention is a lookup by similarity.** Each word asks a question; every word advertises itself;
  the dot product decides who answers; softmax turns that into shares; the word takes that share of
  the answers. Four lines of arithmetic.
- **Order has to be added in,** because "multiply and add" does not care what came first.
- **Several heads** just means doing it more than once with different questions, then mixing.
- **The rest of the block** is plumbing so that deep stacks work: residuals so you do not lose what
  you had, LayerNorm so the numbers stay sane, and a small network with ReLU so it can compute
  instead of only averaging.
- **Learning is not a different thing.** It is `loss → slope → step`, repeated. You did it by hand
  in cell 11 and the model learned where to look.

## Honest notes about this notebook

1. **Every matrix was picked by me** (cells 5-9), except the query in cell 11 which was learned. So
   the attention patterns you saw are *illustrations of the mechanism*, not a trained model's
   behaviour. Cell 11 is the only cell where numbers were found by arithmetic.
2. **2 numbers per word is a toy.** With 2 dimensions, LayerNorm preserves only the ordering of the
   two numbers, so `cat` and `sat` collapsed to the same vector in cell 9b. That is a dimension
   artifact, not the idea; with 512 numbers it does not happen. Toy in, quirk out — I left it visible
   rather than hiding it behind more code.
3. **No decoder, no translation, no training loop over data.** The decoder is the same machine with
   one extra attention layer that looks at the encoder's output. That is in
   `the_cat_sat_transformer.ipynb` (61k parameters, trains on 20 pairs, gets 20/20).
4. **Nothing here was fast.** Real training uses the chain rule to get every slope in one backward
   pass, plus GPUs, plus batching. The maths is the maths you just did.

## Where to go next, in this project

| file | what it adds |
|---|---|
| `the_cat_sat_transformer.ipynb` | the same ideas with PyTorch, trained, with attention heatmaps |
| `hidden_states_and_attention.ipynb` | encoder → hidden state → decoder, and attention you write yourself |
| `streaming_state_no_context_window.ipynb` | what to do when the sentence never ends |
| `video/how_transformers_work_1080p.mp4` | the same story as an animated 5-minute video |

**If you remember one sentence from all of this:** attention is each word deciding which other words
to average itself with, where "which" is a dot product turned into percentages by softmax — and the
model learns the matrices that make those dot products worth something.

_State: done. Runs top to bottom in seconds, numpy only, no other imports than numpy._
